# Stage 2 — Fine-tuning sur annotations manuelles (SSP Cloud)

**Objectif** : reprendre le modèle du Stage 1 et le fine-tuner sur les **46 docs annotés manuellement** pour le calibrer sur de la gold data.

**Évaluation finale** : 50 docs annotés manuellement sur les années **2015-2020** → test de généralisation temporelle.

```
camembert-base
      ↓ Stage 1 (6936 docs bruités 1973/1978)
  models/stage1_model  ←── on part de là
      ↓ Stage 2 (46 docs gold 1973-1993)
  models/stage2_model
      ↓ Évaluation
  data/splits/test_after_2000.json (50 docs, 2015-2020)
```

| Dataset | Docs | Années | Source |
|---------|-----:|--------|-------|
| train | 46 | 1973-1993 | annotations manuelles |
| val | 30 | 1973-1993 | annotations manuelles |
| test | 50 | 2015-2020 | annotations manuelles |

**Prérequis** : avoir lancé `scripts/to_bio.py` pour générer `data/bio_manual/` et avoir le `models/stage1_model/` issu du Stage 1.

In [ ]:
import os, json
import numpy as np
import torch
from pathlib import Path
from collections import defaultdict
import matplotlib.pyplot as plt

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict

print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Chemins locaux SSP Cloud ──────────────────────────────────────────────────
# Adapter PROJECT_DIR si le dépôt est monté ailleurs
PROJECT_DIR = Path("/home/onyxia/work/Named_Entity_Extraction_Archelec_Corpus/projet_archelec")

BIO_DIR      = PROJECT_DIR / "data" / "bio"   # généré par scripts/to_bio.py
MODELS_DIR   = PROJECT_DIR / "models"
RESULTS_DIR  = PROJECT_DIR / "data" / "results" / "CamemBERT"
STAGE1_MODEL = MODELS_DIR / "stage1_model"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Vérification stage1_model
if STAGE1_MODEL.exists():
    fichiers = list(STAGE1_MODEL.iterdir())
    print(f'OK  stage1_model ({len(fichiers)} fichiers)')
else:
    print(f'MANQUANT  {STAGE1_MODEL}')
    print('  -> Lance d\'abord stage1_distantsup_sspcloud.ipynb')

# Vérification des données bio
for fname in ['train.json', 'val.json', 'test.json', 'label2id.json']:
    fpath = BIO_DIR / fname
    if fpath.exists():
        taille = fpath.stat().st_size / 1024 / 1024
        print(f'OK  {fname} ({taille:.2f} MB)')
    else:
        print(f'MANQUANT  {fpath}')
        print('  -> Lance d\'abord : python scripts/to_bio.py')

In [ ]:
# ── Chargement label mapping et splits ───────────────────────────────────────
with open(BIO_DIR / 'label2id.json') as f:
    LABEL2ID = json.load(f)
ID2LABEL   = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)
print(f'Labels ({NUM_LABELS}) : {LABEL2ID}')

def charger_split(chemin):
    with open(chemin, encoding='utf-8') as f:
        data = json.load(f)
    return Dataset.from_list([
        {
            'input_ids': d['input_ids'],
            'ner_tags':  d['ner_tags'],
            'id':        d['id'],
            'annee':     d['annee']
        }
        for d in data
    ])

dataset = DatasetDict({
    'train': charger_split(BIO_DIR / 'train.json'),
    'val':   charger_split(BIO_DIR / 'val.json'),
    'test':  charger_split(BIO_DIR / 'test.json'),
})

print(f'Train : {len(dataset["train"])} docs (1973-1993)')
print(f'Val   : {len(dataset["val"])} docs (1973-1993)')
print(f'Test  : {len(dataset["test"])} docs (2015-2020) <- generalisation temporelle')

tag_counts = defaultdict(int)
for ex in dataset['train']:
    for tag in ex['ner_tags']:
        tag_counts[ID2LABEL[tag]] += 1
entite_counts = {k: v for k, v in tag_counts.items() if k != 'O'}
print('\nEntites dans le train :', entite_counts)

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────────
MAX_LENGTH   = 512
PAD_TOKEN_ID = 1

def preprocess_batch(examples):
    batch_input_ids      = []
    batch_attention_mask = []
    batch_labels         = []
    for input_ids, ner_tags in zip(examples['input_ids'], examples['ner_tags']):
        input_ids = input_ids[:MAX_LENGTH]
        ner_tags  = ner_tags[:MAX_LENGTH]
        pad_len        = MAX_LENGTH - len(input_ids)
        attention_mask = [1] * len(input_ids) + [0] * pad_len
        input_ids      = input_ids + [PAD_TOKEN_ID] * pad_len
        labels         = ner_tags  + [-100] * pad_len
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(labels)
    return {
        'input_ids':      batch_input_ids,
        'attention_mask': batch_attention_mask,
        'labels':         batch_labels,
    }

dataset_proc = dataset.map(
    preprocess_batch,
    batched=True,
    remove_columns=['ner_tags', 'id', 'annee']
)
dataset_proc.set_format('torch')
print('Dataset pret')

In [ ]:
# ── Fonction de métriques ─────────────────────────────────────────────────────
def extraire_spans(tags):
    spans = set()
    i = 0
    while i < len(tags):
        tag = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
        if tag.startswith('B-'):
            entite = tag[2:]
            debut  = i
            i += 1
            while i < len(tags):
                t = tags[i] if isinstance(tags[i], str) else ID2LABEL[tags[i]]
                if t == f'I-{entite}':
                    i += 1
                else:
                    break
            spans.add((entite, debut, i))
        else:
            i += 1
    return spans

def compute_metrics(pred):
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)
    stats        = defaultdict(lambda: {'tp': 0, 'fp': 0, 'fn': 0})
    stats_global = {'tp': 0, 'fp': 0, 'fn': 0}
    for pred_seq, label_seq in zip(predictions, labels):
        pred_clean  = [ID2LABEL[p] for p, l in zip(pred_seq, label_seq) if l != -100]
        label_clean = [ID2LABEL[l] for l in label_seq if l != -100]
        spans_pred  = extraire_spans(pred_clean)
        spans_gold  = extraire_spans(label_clean)
        for span in spans_pred:
            if span in spans_gold:
                stats[span[0]]['tp'] += 1
                stats_global['tp']   += 1
            else:
                stats[span[0]]['fp'] += 1
                stats_global['fp']   += 1
        for span in spans_gold:
            if span not in spans_pred:
                stats[span[0]]['fn'] += 1
                stats_global['fn']   += 1
    def prf(tp, fp, fn):
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0
        r  = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2*p*r / (p+r)  if (p + r) > 0 else 0
        return p*100, r*100, f1*100
    res = {}
    for entite in ['PER', 'ORG', 'LOC', 'MISC']:
        s = stats[entite]
        p, r, f1 = prf(s['tp'], s['fp'], s['fn'])
        res[f'f1_{entite}']        = round(f1, 2)
        res[f'precision_{entite}'] = round(p,  2)
        res[f'recall_{entite}']    = round(r,  2)
    p_g, r_g, f1_g = prf(stats_global['tp'], stats_global['fp'], stats_global['fn'])
    res['f1_global']        = round(f1_g, 2)
    res['precision_global'] = round(p_g,  2)
    res['recall_global']    = round(r_g,  2)
    return res

print('compute_metrics defini')

In [ ]:
# ── Chargement du modèle Stage 1 ─────────────────────────────────────────────
print(f'Chargement {STAGE1_MODEL}...')
model = AutoModelForTokenClassification.from_pretrained(
    str(STAGE1_MODEL),
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True
)
print('Modele Stage 1 charge !')
print(f'Parametres : {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── Fine-tuning Stage 2 sur annotations manuelles ────────────────────────────
# lr plus faible (1e-5) car le modele est deja specialise
# batch size plus petit car peu de donnees (46 docs)
# patience plus haute car peu de donnees = courbe d apprentissage plus lente
USE_FP16 = torch.cuda.is_available()
print(f'fp16 : {USE_FP16}')

args = TrainingArguments(
    output_dir                  = str(MODELS_DIR / 'stage2_checkpoints'),
    num_train_epochs            = 20,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 16,
    learning_rate               = 1e-5,
    weight_decay                = 0.01,
    warmup_ratio                = 0.1,
    eval_strategy               = 'epoch',
    save_strategy               = 'epoch',
    load_best_model_at_end      = True,
    metric_for_best_model       = 'f1_global',
    greater_is_better           = True,
    logging_steps               = 10,
    fp16                        = USE_FP16,
    report_to                   = 'none',
    save_total_limit            = 1,
)

trainer = Trainer(
    model           = model,
    args            = args,
    train_dataset   = dataset_proc['train'],
    eval_dataset    = dataset_proc['val'],
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=5)]
)

print('Fine-tuning Stage 2 sur 46 docs manuels...')
trainer.train()
print('Fine-tuning termine !')

In [ ]:
# ── Évaluation sur test 2015-2020 ─────────────────────────────────────────────
print('=== EVALUATION STAGE 2 sur TEST (2015-2020) ===')
print('Test de generalisation temporelle : le modele na jamais vu ces annees')
print()

results_stage2 = trainer.evaluate(dataset_proc['test'])

print('{:<10} {:>12} {:>10} {:>10}'.format('Entite', 'Precision', 'Recall', 'F1'))
print('-' * 45)
for entite in ['PER', 'ORG', 'LOC', 'MISC']:
    p  = results_stage2.get('eval_precision_' + entite, 0)
    r  = results_stage2.get('eval_recall_'    + entite, 0)
    f1 = results_stage2.get('eval_f1_'        + entite, 0)
    print('{:<10} {:>11.2f}% {:>9.2f}% {:>9.2f}%'.format(entite, p, r, f1))
print('-' * 45)
p_g  = results_stage2.get('eval_precision_global', 0)
r_g  = results_stage2.get('eval_recall_global',    0)
f1_g = results_stage2.get('eval_f1_global',        0)
print('{:<10} {:>11.2f}% {:>9.2f}% {:>9.2f}%'.format('GLOBAL', p_g, r_g, f1_g))

results_path = RESULTS_DIR / 'stage2_metrics_test.json'
with open(results_path, 'w') as f:
    json.dump(results_stage2, f, indent=2)
print(f'\nResultats sauvegardes dans {results_path}')

In [ ]:
# ── Sauvegarde modele Stage 2 ─────────────────────────────────────────────────
stage2_model_dir = MODELS_DIR / 'stage2_model'
trainer.save_model(str(stage2_model_dir))
print(f'Modele Stage 2 sauvegarde dans {stage2_model_dir}')
print()
print('Fichiers du modele :')
for fname in sorted(stage2_model_dir.iterdir()):
    taille = fname.stat().st_size / 1024 / 1024
    print(f'  {fname.name} ({taille:.1f} MB)')

In [ ]:
# ── Analyse des prédictions sur le test 2015-2020 ─────────────────────────────
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

with open(BIO_DIR / 'test.json', encoding='utf-8') as f:
    test_raw = json.load(f)

print('=== EXEMPLES DE PREDICTIONS SUR 2015-2020 ===')
for doc in test_raw[:3]:
    input_ids      = torch.tensor([doc['input_ids'][:512]], dtype=torch.long).to(device)
    attention_mask = torch.ones_like(input_ids).to(device)
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    preds  = torch.argmax(logits, dim=-1)[0].cpu().tolist()
    tokens = doc['tokens'][:512]
    gold   = [ID2LABEL[t] for t in doc['ner_tags'][:512]]
    pred_labels = [ID2LABEL[p] for p in preds]

    print(f'\n--- ID: {doc["id"]} ({doc["annee"]}) ---')
    print('{:<20} {:>10} {:>10} {:>8}'.format('Token', 'Gold', 'Pred', 'Match'))
    print('-' * 52)
    n = 0
    for token, g, p in zip(tokens, gold, pred_labels):
        if g != 'O' or p != 'O':
            match = 'OK' if g == p else 'XX'
            token_clean = token.replace('\u2581', ' ').strip()
            print('  {:<18} {:>10} {:>10} {:>8}'.format(token_clean, g, p, match))
            n += 1
            if n >= 15:
                print('  [... tronque ...]')
                break
    if n == 0:
        print('  (aucune entite dans ce document)')